In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Zacznij pracę z cięciem Circuit przy użyciu cięć przewodów


<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie był tworzony z użyciem poniższych wymagań.
Zalecamy korzystanie z tych wersji lub nowszych.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
qiskit-addon-cutting~=0.10.0
```
</details>
Ten przewodnik demonstruje działający przykład cięć przewodów z użyciem pakietu `qiskit-addon-cutting`. Obejmuje rekonstrukcję wartości oczekiwanych siedmio-Qubitowego Circuit przy użyciu cięcia przewodów.

Cięcie przewodu jest w tym pakiecie reprezentowane jako dwu-Qubitowa instrukcja [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move), która jest zdefiniowana jako reset drugiego Qubitu, na którym działa instrukcja, po którym następuje zamiana obu Qubitów. Ta operacja jest równoważna przeniesieniu stanu pierwszego Qubitu do drugiego Qubitu, przy jednoczesnym odrzuceniu przychodzącego stanu drugiego Qubitu.

Pakiet jest zaprojektowany tak, aby był spójny ze sposobem, w jaki musisz traktować cięcia przewodów podczas działania na fizycznych Qubitach. Na przykład, cięcie przewodu może wziąć stan fizycznego Qubitu $n$ i kontynuować go jako fizyczny Qubit $m$ po cięciu. Możesz myśleć o „cięciu instrukcji" jako o ujednoliconym frameworku do rozważania zarówno cięć przewodów, jak i bramek w tym samym formalizmie (ponieważ cięcie przewodu to po prostu cięcie instrukcji [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move)). Używanie tego frameworku do cięcia przewodów umożliwia również ponowne użycie Qubitów, co jest wyjaśnione w sekcji dotyczącej [ręcznego cięcia przewodów](#cut-wires-using-the-low-level-move-instruction).

Jednobitowa instrukcja [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) działa jako bardziej abstrakcyjny, prostszy interfejs do pracy z cięciami przewodów. Pozwala oznaczyć miejsca w Circuit, gdzie przewód powinien być przecięty na wysokim poziomie, a następnie dodatek do cięcia Circuit wstawia odpowiednie instrukcje [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) za ciebie.

Poniższy przykład demonstruje rekonstrukcję wartości oczekiwanej po cięciu przewodów. Stworzysz Circuit z kilkoma nielokalnych bramkami i zdefiniujesz obserwowalnych do oszacowania.

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit_ibm_runtime import SamplerV2, Batch
from qiskit_aer.primitives import EstimatorV2

from qiskit_addon_cutting.instructions import Move, CutWire
from qiskit_addon_cutting import (
    partition_problem,
    generate_cutting_experiments,
    cut_wires,
    expand_observables,
    reconstruct_expectation_values,
)


qc_0 = QuantumCircuit(7)
for i in range(7):
    qc_0.rx(np.pi / 4, i)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)
qc_0.cx(3, 4)
qc_0.cx(3, 5)
qc_0.cx(3, 6)
qc_0.cx(0, 3)
qc_0.cx(1, 3)
qc_0.cx(2, 3)

# Define observable
observable = SparsePauliOp(["ZIIIIII", "IIIZIII", "IIIIIIZ"])

# Draw circuit
qc_0.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg" alt="Output of the previous code cell" />

![Wyjście poprzedniej komórki kodu](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/b481ef2d-3912-4eac-9755-335e8f5db886-0.svg)

## Tnij przewody przy użyciu wysokopoziomowej instrukcji `CutWire`
Następnie wykonaj cięcia przewodów przy użyciu jednobitowej instrukcji [`CutWire`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-cut-wire) na Qubicie $q_3$. Gdy podeksperymenty będą gotowe do wykonania, użyj funkcji [`cut_wires()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#cut_wires), aby przekształcić `CutWire` w instrukcje [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move) na nowo przydzielonych Qubitach.

In [2]:
qc_1 = QuantumCircuit(7)
for i in range(7):
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(CutWire(), [3])
qc_1.cx(3, 4)
qc_1.cx(3, 5)
qc_1.cx(3, 6)
qc_1.append(CutWire(), [3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg" alt="Output of the previous code cell" />

![Wyjście poprzedniej komórki kodu](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/9bac1915-316b-49d0-a1a1-145047679530-0.svg)

> **Info:** Gdy Circuit jest rozszerzany przez jedno lub więcej cięć przewodów, oberwowalna musi zostać zaktualizowana, aby uwzględnić dodatkowe Qubity, które są wprowadzane. Pakiet `qiskit-addon-cutting` posiada wygodną funkcję [`expand_observables()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#expand_observables), która przyjmuje obiekty `PauliList` oraz oryginalne i rozszerzone Circuit jako argumenty i zwraca nową `PauliList`.
> 
>     Zwrócona `PauliList` nie będzie zawierała żadnych informacji o współczynnikach oryginalnej obserwowalnej, ale można je pominąć do momentu rekonstrukcji ostatecznej wartości oczekiwanej.

In [3]:
# Transform CutWire instructions to Move instructions
qc_2 = cut_wires(qc_1)

# Expand the observable to match the new circuit size
expanded_observable = expand_observables(observable.paulis, qc_0, qc_2)
print(f"Expanded Observable: {expanded_observable}")
qc_2.draw("mpl")

Expanded Observable: ['ZIIIIIIII', 'IIIZIIIII', 'IIIIIIIIZ']


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d398b397-0167-4db9-97ae-6ea502dc43e3-1.svg" alt="Output of the previous code cell" />

### Partition the circuit and observable

Now the problem can be separated into partitions. This is accomplished using the [`partition_problem()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#partition_problem) function with an optional set of partition labels to specify how to separate the circuit. Qubits sharing a common partition label are grouped together, and any non-local gates spanning more than one partition are cut.

If no partition labels are provided, then the partitioning will be automatically determined based on the connectivity of the circuit. Read the next section on [cutting wires manually](#cut-wires-using-the-low-level-move-instruction) for more information on including partition labels.

In [4]:
partitioned_problem = partition_problem(
    circuit=qc_2,
    observables=expanded_observable,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits[0].draw("mpl")

Subobservables to measure: 
{0: PauliList(['IIIII', 'ZIIII', 'IIIIZ']), 1: PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/5fb034f2-da8a-4f4d-ab9b-c990593e04fc-1.svg" alt="Output of the previous code cell" />

In [5]:
subcircuits[1].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg" alt="Output of the previous code cell" />

In this partitioning scheme, you have cut two wires, resulting in a sampling overhead of $4^4$.

### Generate subexperiments to execute and post-process results

To estimate the expectation value of the full-sized circuit, several subexperiments are generated from the decomposed gates' joint quasi-probability distribution and then executed on one (or more) QPUs. The [`generate_cutting_experiments`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) method does this by ingesting arguments for the `subcircuits` and `subobservables` dictionaries you created above, as well as for the number of samples to take from the distribution.

<Admonition type="note" title="Note about the number of samples">
    The `num_samples` argument specifies how many samples to draw from the quasi-probability distribution and determines the accuracy of the coefficients used for the reconstruction. Passing infinity (`np.inf`) ensures all coefficients are calculated exactly. Read the API docs on [generating weights](/docs/api/qiskit-addon-cutting/qpd#generate_qpd_weights) and [generating cutting experiments](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) for more information.
</Admonition>

In [6]:
# Generate subexperiments
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits, observables=subobservables, num_samples=np.inf
)

# Set a backend to use and transpile the subexperiments
backend = FakeManilaV2()
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
isa_subexperiments = {
    label: pass_manager.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Submit each partition's subexperiments to the Qiskit Runtime Sampler
# primitive, in a single batch so that the jobs will run back-to-back.
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
    # Retrieve results
    results = {label: job.result() for label, job in jobs.items()}

Lastly, the expectation value of the full circuit can be reconstructed using the [`reconstruct_expectation_values()`](/docs/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values) method.


The code block below reconstructs the results and compares them with the exact expectation value.

In [7]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
# Apply the coefficients of the original observable
reconstructed_expval = np.dot(reconstructed_expval_terms, observable.coeffs)


# Compute the exact expectation value using the `qiskit_aer` package.
estimator = EstimatorV2()
exact_expval = estimator.run([(qc_0, observable)]).result()[0].data.evs
print(
    f"Reconstructed expectation value: {np.real(np.round(reconstructed_expval, 8))}"
)
print(f"Exact expectation value: {np.round(exact_expval, 8)}")
print(
    f"Error in estimation: {np.real(np.round(reconstructed_expval-exact_expval, 8))}"
)
print(
    f"Relative error in estimation: {np.real(np.round((reconstructed_expval-exact_expval) / exact_expval, 8))}"
)

Reconstructed expectation value: 1.45965266
Exact expectation value: 1.59099026
Error in estimation: -0.1313376
Relative error in estimation: -0.08255085


![Wyjście poprzedniej komórki kodu](../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/d0e86f81-7c7e-4ccf-951c-9cd039135dc9-0.svg)

W tym schemacie podziału przecięto dwa przewody, co skutkuje narzutem próbkowania $4^4$.

### Generuj podeksperymenty do wykonania i przetwarzaj wyniki

Aby oszacować wartość oczekiwaną pełnego Circuit, kilka podeksperymentów jest generowanych ze wspólnego quasi-rozkładu prawdopodobieństwa rozłożonych bramek, a następnie wykonywanych na jednym (lub kilku) QPU. Metoda [`generate_cutting_experiments`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments) robi to, przyjmując argumenty dla słowników `subcircuits` i `subobservables`, które stworzyłeś powyżej, a także dla liczby próbek pobieranych z rozkładu.

> **Note:** Argument `num_samples` określa, ile próbek należy pobrać z quasi-rozkładu prawdopodobieństwa i wyznacza dokładność współczynników używanych do rekonstrukcji. Przekazanie nieskończoności (`np.inf`) zapewni dokładne obliczenie wszystkich współczynników. Przeczytaj dokumentację API na temat [generowania wag](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qpd#generate_qpd_weights) i [generowania eksperymentów cięcia](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#generate_cutting_experiments), aby uzyskać więcej informacji.

In [8]:
qc_1 = QuantumCircuit(8)
for i in [*range(4), *range(5, 8)]:
    qc_1.rx(np.pi / 4, i)
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)
qc_1.append(Move(), [3, 4])
qc_1.cx(4, 5)
qc_1.cx(4, 6)
qc_1.cx(4, 7)
qc_1.append(Move(), [4, 3])
qc_1.cx(0, 3)
qc_1.cx(1, 3)
qc_1.cx(2, 3)

# Expand observable
observable_expanded = SparsePauliOp(["ZIIIIIII", "IIIIZIII", "IIIIIIIZ"])
qc_1.draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/15461a2c-85a9-4cb2-a632-b9597ccbc4bd-0.svg" alt="Output of the previous code cell" />

Na koniec wartość oczekiwana pełnego Circuit może być zrekonstruowana przy użyciu metody [`reconstruct_expectation_values()`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/qiskit-addon-cutting#reconstruct_expectation_values).

Poniższy blok kodu rekonstruuje wyniki i porównuje je z dokładną wartością oczekiwaną.

In [9]:
partitioned_problem = partition_problem(
    circuit=qc_1,
    partition_labels="AAAABBBB",
    observables=observable_expanded.paulis,
)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables
bases = partitioned_problem.bases

print(f"Subobservables to measure: \n{subobservables}\n")
print(f"Sampling overhead: {np.prod([basis.overhead for basis in bases])}")
subcircuits["A"].draw("mpl")

Subobservables to measure: 
{'A': PauliList(['IIII', 'ZIII', 'IIIZ']), 'B': PauliList(['ZIII', 'IIII', 'IIII'])}

Sampling overhead: 256.0


<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/2139745a-bdc3-40bd-bd6f-d26d2a5b5b14-1.svg" alt="Output of the previous code cell" />

In [10]:
subcircuits["B"].draw("mpl")

<Image src="../docs/images/guides/qiskit-addons-cutting-wires/extracted-outputs/4aeb3f1f-a55e-49c4-a7bd-837132429ee1-0.svg" alt="Output of the previous code cell" />

> **Caution:** Aby dokładnie zrekonstruować wartość oczekiwaną, współczynniki oryginalnej obserwowalnej (które różnią się od wyjścia `generate_cutting_experiments()`) muszą być zastosowane do wyjścia rekonstrukcji, ponieważ ta informacja została utracona podczas generowania eksperymentów cięcia lub gdy oberwowalna była rozwijana.
> 
>     Zazwyczaj te współczynniki można zastosować przez `numpy.dot()`, jak pokazano wcześniej.
## Tnij przewody przy użyciu niskopoziomowej instrukcji `Move`
Jednym z ograniczeń używania wyższopoziomowej instrukcji `CutWire` jest to, że nie pozwala na ponowne użycie Qubitów. Jeśli jest to pożądane w eksperymencie cięcia, możesz zamiast tego ręcznie umieszczać instrukcje [`Move`](https://docs.quantum.ibm.com/api/qiskit-addon-cutting/instructions-move). Ponieważ jednak instrukcja `Move` odrzuca stan docelowego Qubitu, ważne jest, aby ten Qubit nie był splątany z resztą systemu. W przeciwnym razie operacja resetu spowoduje częściowe zapaśnięcie stanu Circuit po cięciu przewodu.

Poniższy blok kodu wykonuje cięcie przewodu na Qubicie $q_3$ dla tego samego przykładowego Circuit co poprzednio. Różnica polega na tym, że możesz ponownie użyć Qubitu, odwracając operację `Move` tam, gdzie wykonano drugie cięcie przewodu (jednak nie zawsze jest to możliwe i zależy od ciętego Circuit).